# Phase 2: Pipeline Validation & Sanity Checks

This notebook comprehensively tests the output of `pipeline_sample.csv` to ensure the data mapping 
and derived features are behaving logically before I begin evaluating models.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add src dynamically
sys.path.append(os.path.abspath('../src'))
import validation_checks

## 1. Global Shape & Types

In [2]:
df = pd.read_csv('../outputs/pipeline_sample.csv')
print(f"Pipeline output shape: {df.shape}\n")
print("Data Types:\n", df.dtypes)

Pipeline output shape: (43028, 39)

Data Types:
 user_id                        int64
video_id                       int64
date                           int64
hourmin                        int64
time_ms                        int64
is_click                       int64
is_like                        int64
is_follow                      int64
is_comment                     int64
is_forward                     int64
is_hate                        int64
long_view                      int64
play_time_ms                   int64
duration_ms                    int64
profile_stay_time              int64
comment_stay_time              int64
is_profile_enter               int64
is_rand                        int64
tab                            int64
author_id                      int64
video_type                    object
upload_dt                     object
upload_type                   object
visible_status               float64
video_duration               float64
server_width              

## 2. Nulls & Distributions

In [3]:
print("Top 10 Null Columns:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nNumeric Summary Stats (Focus columns):")
cols = ['implicit_completion_ratio', 'item_age_days', 'play_time_ms', 'duration_ms']
print(df[cols].describe())

print("\ny_relevant Distribution:")
print(df['y_relevant'].value_counts(normalize=True))

Top 10 Null Columns:
video_duration    1310
music_type        1077
user_id              0
hourmin              0
time_ms              0
video_id             0
date                 0
is_like              0
is_click             0
is_follow            0
dtype: int64

Numeric Summary Stats (Focus columns):


       implicit_completion_ratio  item_age_days   play_time_ms   duration_ms
count               43028.000000   43028.000000   43028.000000  4.302800e+04
mean                    0.129643      22.505680    6725.488054  1.053191e+05
std                     0.266147       4.666312   17758.981998  1.049446e+05
min                     0.000000      11.194588       0.000000  0.000000e+00
25%                     0.012222      19.277269    1328.000000  2.843300e+04
50%                     0.032127      23.476551    2082.000000  7.740000e+04
75%                     0.104157      26.336760    4542.250000  1.462660e+05
max                     2.000000      29.660845  508072.000000  1.177720e+06

y_relevant Distribution:
y_relevant
0    0.948894
1    0.051106
Name: proportion, dtype: float64


## 3. Session Statistics

In [4]:
sess_lengths = df.groupby('session_id').size()
print("Session Length Distribution:")
print(sess_lengths.describe())

# Flag overly massive sessions (e.g. > 500 impressions in a single 30-min bounded timeframe)
print(f"\nSessions with > 200 items: {(sess_lengths > 200).sum()}")

Session Length Distribution:
count    19045.000000
mean         2.259281
std          3.109242
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        220.000000
dtype: float64

Sessions with > 200 items: 1


## 4. Logical Contradiction Checking

In [5]:
masks = validation_checks.check_logical_contradictions(df)

print("Contradiction Report (Violation Counts):\n")
for key, mask in masks.items():
    print(f"  - {key}: {mask.sum()} flags")

Contradiction Report (Violation Counts):

  - hate_but_relevant: 0 flags
  - invalid_duration: 1310 flags
  - high_completion_ratio: 277 flags
  - exact_duplicate_event: 2 flags
  - missing_upload_dt: 0 flags


## 5. Explicit Edge Case Review Output

In [6]:
review_df = validation_checks.sample_edge_cases(df, masks)

cols_to_view = ['user_id', 'video_id', 'time_ms', 'session_id', 'is_hate', 'y_relevant', 
                'duration_ms', 'play_time_ms', 'implicit_completion_ratio', 
                'upload_dt', 'item_age_days', 'is_edge_case', 'edge_case_reason']

output_path = '../outputs/validation_sample_rows.csv'
review_df[cols_to_view].to_csv(output_path, index=False)
print(f"Exported {len(review_df)} sample rows (mixed edge cases and normals) to {output_path}.")

Exported 200 sample rows (mixed edge cases and normals) to ../outputs/validation_sample_rows.csv.
